In [ ]:
#data reading and plotting libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
import keras

In [ ]:
dir = glob('/kaggle/input/plant-diseases/dataset_itr2/train/*')
dir

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

image_size = (64, 64)

train_dir = '/kaggle/input/plant-diseases/dataset_itr2/train/'
data_generator = ImageDataGenerator(rescale=1.0/255.0,validation_split = 0.3)

train_generator = data_generator.flow_from_directory(train_dir,
                                                      target_size=image_size,
                                                      batch_size=200,
                                                      class_mode='categorical',
                                                      subset='training',
                                                      shuffle=True)

val_generator = data_generator.flow_from_directory(train_dir,
                                                      target_size=image_size,
                                                      batch_size=100,
                                                      class_mode='categorical',
                                                      subset='validation',
                                                      shuffle=True)

In [ ]:
len(train_generator),len(val_generator)

### Build CNN Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout,BatchNormalization
from tensorflow.keras.regularizers import l2,l1

In [ ]:
model = Sequential()

In [ ]:
input_shape = (64, 64, 3)

# First Convolutional Layer
model.add(Conv2D(32, (3, 3), activation='relu',kernel_regularizer=l2(0.001), input_shape=input_shape))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))


# Second Convolutional Layer
model.add(Conv2D(64, (3, 3), activation='relu',kernel_regularizer=l2(0.001)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))


# third Convolutional Layer
model.add(Conv2D(128, (3, 3), activation='relu',kernel_regularizer=l2(0.001)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(256, (3, 3), activation='relu',kernel_regularizer=l2(0.001)))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))


# Flatten the feature maps
model.add(Flatten())

# Fully Connected Layers
model.add(Dense(512, activation='relu',kernel_regularizer=l2(0.001)))
model.add(BatchNormalization())
model.add(Dense(38, activation='softmax'))


In [ ]:
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
from tensorflow.keras.utils import plot_model

plot_model(model, to_file='model.png', show_shapes=True)

In [ ]:
model.fit(train_generator, epochs=10, validation_data=val_generator)

In [ ]:
model.fit(train_generator, epochs=25, validation_data=val_generator)

In [ ]:
model.fit(train_generator, epochs=30,validation_data=val_generator)

In [ ]:
model.fit(train_generator, epochs=10 ,validation_data=val_generator)

In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(model.history.history['accuracy'], label='Train Accuracy')
plt.plot(model.history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(model.history.history['loss'], label='Train Loss')
plt.plot(model.history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()


plt.tight_layout()
plt.show()

In [ ]:
model.save('CNN_Model64.h5')

In [ ]:
dir_image = glob('/kaggle/input/plant-diseases/dataset_itr2/train/Tomato___Late_blight/*')
dir_image[5]

In [ ]:
import cv2
image = plt.imread(dir_image[5])
resize_shape = (64,64)
resized = cv2.resize(image,(64,64),interpolation = cv2.INTER_AREA)
plt.imshow(resized)
plt.axis('off')

In [ ]:
predictions = model.predict(val_generator)

In [ ]:
diseases_labels = []

for key, value in train_generator.class_indices.items():
   diseases_labels.append(key)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

def evaluate(actual, predictions):
  pre = []
  for i in predictions:
    pre.append(np.argmax(i))

  accuracy = (pre == actual).sum() / actual.shape[0]
  print(f'Accuracy: {accuracy}')
    

  fig, ax = plt.subplots(figsize=(20,20))
  conf_mat = confusion_matrix(actual, pre)
  sns.heatmap(conf_mat, annot=True, fmt='.0f', cmap="YlGnBu", xticklabels=diseases_labels, yticklabels=diseases_labels).set_title('Confusion Matrix Heat map')
  plt.show()

In [ ]:
confusion_matrix(val_generator.classes, predictions)